# Process the goggles-left (single-eye) session

Builds the eye-conditioned `MonocularDevice` for the **left-eye** session
`260722_SAM045_goggles-left`, runs `SessionAnalyzer.analyze_session_stats`
and caches `{session_name}_dataframe.pkl` next to the session data (same
pattern as `process-monocular-session.ipynb`), then:

1. plots each **left-reliable** cell's response (`CellVisualizer.make_eye_plot`), and
2. plots the histogram of reliable cells' receptive-field positions (`SessionVisualizer.plot_rf_histograms`).

Run this in its own kernel (`distance-project` env) with **only** `disparity-analysis` on `sys.path`.


In [ ]:
import sys, os
import pandas as pd
import matplotlib.pyplot as plt

base_dir = r"C:\Users\acostamendoza\projects\distance-project"
for p in (base_dir, base_dir + r"\disparity-analysis"):
    if p not in sys.path:
        sys.path.append(p)

from src.data.MonocularDevice import MonocularDevice
from src.analysis.CellAnalysis import SessionAnalyzer
from src.viz.SessionVisualizer import SessionVisualizer
from src.viz.CellVisualizer import CellVisualizer

%load_ext autoreload
%autoreload 2

In [ ]:
mouse, date, experiment = "SAM045", "260722", "goggles-left"
M = MonocularDevice(mouse_name=mouse, record_date=date, experiment_name=experiment)
print("session:", M.session_name, "| eyes:", M.present_eyes,
      "| single_eye:", M.single_eye, "| cells:", len(M.suite2p_id))

In [ ]:
# load cached stats if present, else compute once (slow) and save the pkl
pkl_path = os.path.join(M.data_dir, f"{M.session_name}_dataframe.pkl")
if os.path.exists(pkl_path):
    stats_df = pd.read_pickle(pkl_path)
    print(f"Loaded cached stats: {pkl_path}")
else:
    stats_df = SessionAnalyzer(M).analyze_session_stats(n_iterations=200)
    SessionAnalyzer(M).save_dataframe(stats_df)
stats_df.head()

In [ ]:
sv = SessionVisualizer(M, stats_df=stats_df)
sets = sv.print_summary()
# single-eye session -> reliable set is keyed 'reliable'; 'left' if both eyes present
left_reliable = sorted(sets.get("left", sets.get("reliable")))
print(f"{len(left_reliable)} left-reliable cells")

In [ ]:
# response of every left-reliable cell (trial heatmap + mean +/- SEM tuning curve)
# For many cells this renders a lot of figures; to save them to disk instead use:
#   CellVisualizer(M).save_all_cells(kind="eye", eye="left")
viz = CellVisualizer(M)
for cell in left_reliable:
    viz.make_eye_plot(cell, eye="left")
    plt.show()

In [ ]:
# histogram of reliable cells and their receptive-field position (px)
sv.plot_rf_histograms()
plt.show()